# Factorial (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Factorial_java.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Factorial`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Factorial_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Factorial_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono campi `static` della classe, e la traccia si attiva con `trace = true`.

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Fattoriale: Factorial

**Input:** un numero naturale $n$. **Output:** $n!$, con $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ passi; $n + 1$ chiamate, profondità $n + 1$ (l'albero è un cammino).

Pseudocodice: sezione 14.4 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
%%writefile Factorial.java
public class Factorial
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Factorial(n) valori corretti solo fino a n = 20 (in C++ e Java i 64 bit traboccano oltre)
    static long factorial(int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) event("Factorial(0): n = 0 => caso base");
            pop();
            if (trace) event("Factorial(0) restituisce 1");
            return 1;
        }
        if (trace) event(String.format("Factorial(%d): %d != 0 => deve calcolare %d * Factorial(%d) e attende", n, n, n, n - 1));
        cnt[5]++;
        long f = factorial(n - 1);
        long r = n * f;
        pop();
        if (trace) event(String.format("Factorial(%d) restituisce %d * %d = %d", n, n, f, r));
        return r;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: Factorial(4), con lo stack
        int n = 4;
        reset();
        trace = true;
        long r = factorial(n);
        trace = false;
        System.out.printf("Factorial(%d) = %d\n", n, r);
        System.out.printf("Chiamate = %d (nodi dell'albero), profondita' della ricorsione = %d\n", calls, maxDepth);
        printSteps(2, 3, 5);
        System.out.printf("S(n) = 2n + 2 = %d\n", 2 * n + 2);

        // Chiamate, profondita' e passi al variare di n
        System.out.printf("\n");
        System.out.printf("  n |                   n! | chiamate | profondita' | S(n) | 2n + 2\n");
        int[] values = {0, 1, 2, 3, 4, 5, 10, 15, 20};
        for (int v : values)
        {
            reset();
            long f = factorial(v);
            System.out.printf("%3d | %20d | %8d | %11d | %4d | %d\n", v, f, calls, maxDepth, total(), 2 * v + 2);
        }
    }
}

Writing Factorial.java


In [3]:
!javac Factorial.java && java Factorial

 1. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [4]
 2. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [4, 3]
 3. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => caso base   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) restituisce 1   stack: [4, 3, 2, 1]
 7. Factorial(1) restituisce 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) restituisce 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) restituisce 3 * 2 = 6   stack: [4]
10. Factorial(4) restituisce 4 * 6 = 24   stack: []
Factorial(4) = 24
Chiamate = 5 (nodi dell'albero), profondita' della ricorsione = 5
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
Passi totali S = 10
S(n) = 2n + 2 = 10

  n |                   n! | chiamate | profondita' | S(n) | 2n + 2
  0 |                    1 |        1 |           1 | 

**Prova tu.** Cambia $n$ e guarda lo stack crescere e poi svuotarsi.

In [4]:
%%writefile ProvaFactorial.java
public class ProvaFactorial
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- cambia n (al massimo 20)
        Factorial.reset();
        Factorial.trace = true;
        long r = Factorial.factorial(n);
        Factorial.trace = false;
        System.out.printf("Factorial(%d) = %d, chiamate = %d, profondita' = %d, S = %d\n", n, r, Factorial.calls, Factorial.maxDepth, Factorial.total());
    }
}

Writing ProvaFactorial.java


In [5]:
!javac ProvaFactorial.java && java ProvaFactorial

 1. Factorial(6): 6 != 0 => deve calcolare 6 * Factorial(5) e attende   stack: [6]
 2. Factorial(5): 5 != 0 => deve calcolare 5 * Factorial(4) e attende   stack: [6, 5]
 3. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => caso base   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) restituisce 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) restituisce 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) restituisce 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) restituisce 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) restituisce 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) restituisce 5 * 24 = 120   stack: [6]
14. Factorial(6) restituisce 6 * 120

---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).